<a href="https://colab.research.google.com/github/Gowtham13042007/cron_job/blob/main/lasso_and_ridge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.linear_model import Ridge,Lasso
from sklearn.model_selection import GridSearchCV,train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.datasets import fetch_california_housing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pandas as pd
from sklearn.impute import SimpleImputer

data=fetch_california_housing()
X=data.data
y=data.target

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

X_train = pd.DataFrame(X_train, columns=data.feature_names)
X_test = pd.DataFrame(X_test, columns=data.feature_names)

numeric_column_transformer=Pipeline(steps=[
    ('impute',SimpleImputer(strategy='mean')),
    ('scale',StandardScaler())
])

preprocessor=ColumnTransformer(transformers=[
    ('numeric',numeric_column_transformer,X_train.columns)
])

ridge_model=Pipeline(steps=[
    ('preprocessor',preprocessor), #Loss=MSE+α∑wi2​
    ('model',Ridge())
])

lasso_model=Pipeline(steps=[
    ('preprocessor',preprocessor),#Loss=MSE+α∑∣wi​∣
    ('model',Lasso())
])

param_grid = {
    'model__alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
}

grid_search = GridSearchCV(
    estimator=ridge_model,
    param_grid=param_grid,     #same u can replace with the lasso_model
    cv=5,
    scoring='r2',
    n_jobs=-1
)

# Fit the grid search to find the best alpha
grid_search.fit(X_train, y_train)

print(f"Best Alpha: {grid_search.best_params_['model__alpha']}")
print(f"Best CV R2 Score: {grid_search.best_score_}")

# Evaluate the best model on the test set
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
print(f"Test Set R2 Score: {r2_score(y_test, y_pred_best)}")
